# Arbiter HR RAG Lab &nbsp;·&nbsp; Kai Components — unstructured policy pipeline

An end-to-end **DIY RAG** lab for the **semantic / unstructured** path: HR-policy **PDFs** →
extract → **chunk** → **Titan v2 embed** → **Amazon S3 Vectors** (`hr-policies` index) →
retrieve (+ metadata filter, + optional rerank) → **grounded answer** → evaluate.

It is the sibling of `sales_rag_lab.ipynb` (the structured/text-to-SQL path). Both import the
same `arbiter_rag` library, so **what you validate here is exactly what the deployed
`hr_specialist` AgentCore agent runs**.

> **Kai Components** is a fictional employee-owned Hawaiian electronics-components retailer —
> the same business the sales RAG answers about. All policy figures are fictional.

**Runs offline.** Keep `RUN_AWS = False` (the default) to execute every cell with local
cross-checks and **no AWS calls / no cost**. Flip it to `True` (with creds + provisioned
resources) to exercise the real embed → S3 Vectors → generate path. See `rag_instructions.md`.

## 0 · Experiment config &nbsp;·&nbsp; edit this one cell, then re-run

In [1]:
# --- Experiment config (edit me) ---------------------------------------------
HR_BUCKET   = "dev-st21arbiter-poc-hr-vectors"   # S3 Vectors bucket (created by the ingest step)
HR_INDEX    = "hr-policies"                       # S3 Vectors index name
TOP_K       = 4              # retrieval depth
CHUNK_STRATEGY = "semantic"  # "semantic" | "fixed" | "recursive"
CHUNK_VERSION  = "v1"        # stamped into chunk ids/metadata (re-ingest idempotency)
USE_RERANK  = False          # Bedrock rerank (needs bedrock:Rerank IAM); off by default

# Guard — keep False to run offline (local only, no AWS calls, no cost).
RUN_AWS     = False          # embed + S3 Vectors ingest/query + generate (Bedrock)

# --- Bootstrap: import the shared library (pip install -e rag_src, or sys.path) ----
import sys, pathlib, json, datetime as dt
_here = pathlib.Path.cwd()
_rag_src = next((p for p in [_here, *_here.parents] if (p / "arbiter_rag").is_dir()), None)
if _rag_src and str(_rag_src) not in sys.path:
    sys.path.insert(0, str(_rag_src))

import arbiter_rag
from arbiter_rag.config import get_settings, DATA_ROOT, RAG_ROOT
from arbiter_rag import loaders, chunking, embeddings, vectors, retrieval, evaluation

S = get_settings()
POLICY_DIR = DATA_ROOT / "Hawaii_HR_Policies"
print("policy dir  :", POLICY_DIR, "| exists:", POLICY_DIR.exists())
print("gen model   :", S.generation_model_id, "  (Amazon Nova 2 Lite; swap via settings.toml)")
print("embed model :", S.embedding_model_id, f"({S.embedding_dim}d)")
print("vector index:", f"{HR_BUCKET}/{HR_INDEX}", "| rerank:", USE_RERANK)
print("guard       : RUN_AWS =", RUN_AWS)

policy dir  : /Users/sridharpchome/Documents/Claude/Demo-ST21Arbiter/data/Hawaii_HR_Policies | exists: True
gen model   : us.amazon.nova-2-lite-v1:0   (Amazon Nova 2 Lite; swap via settings.toml)
embed model : amazon.titan-embed-text-v2:0 (1024d)
vector index: dev-st21arbiter-poc-hr-vectors/hr-policies | rerank: False
guard       : RUN_AWS = False


## 1 · Load the HR policy corpus (UNSTRUCTURED)

Six deterministic policy PDFs. `iter_hr_documents` parses `doc_id` + `policy_category` from
each filename (`HR-LEAVE-001_leave.pdf`) and `title` / `effective_date` from the body — the
exact fields the `hr-policies` index metadata schema expects. If the corpus is missing, the
generator (`rag_src/data_generators/gen_hr_pdfs.py`) is run for you.

In [2]:
if not POLICY_DIR.exists() or not list(POLICY_DIR.glob("*.pdf")):
    print("corpus missing — generating it…")
    from data_generators import gen_hr_pdfs
    gen_hr_pdfs.write_pdfs()

docs = loaders.iter_hr_documents(POLICY_DIR)
print(f"{len(docs)} policy documents loaded\n")
for d in docs:
    print(f"  {d['doc_id']:14} {d['policy_category']:12} eff={d['effective_date']}  {d['title']}")

6 policy documents loaded

  HR-BEN-002     benefits     eff=2025-01-01  Employee Benefits Policy
  HR-COMP-003    compensation eff=2025-02-01  Sales Compensation and Commission Policy
  HR-CONDUCT-004 conduct      eff=2025-01-01  Code of Conduct and Store Standards
  HR-LEAVE-001   leave        eff=2025-01-01  Paid Time Off and Leave Policy
  HR-PAY-005     payroll      eff=2025-01-01  Payroll and Scheduling Policy
  HR-PERK-006    perks        eff=2025-03-01  Employee Perks and Discounts Policy


## 2 · Chunk the documents

`chunk_document` splits each policy on section/paragraph boundaries (the `semantic` strategy)
and copies the filterable metadata (`policy_category`, `title`, `effective_date`, `doc_id`) onto
every chunk. `chunking_version` is baked into each chunk id so re-ingesting the same version is
idempotent and deletable. **Experiment:** change `CHUNK_STRATEGY` / `CHUNK_VERSION` in cell 0.

In [3]:
def to_epoch_days(iso):
    return (dt.date.fromisoformat(iso) - dt.date(1970, 1, 1)).days if iso else 0

all_chunks = []
for d in docs:
    base_meta = {
        "policy_category": d["policy_category"], "title": d["title"],
        "source_uri": d["source_uri"], "effective_date": d["effective_date"],
        "effective_epoch": to_epoch_days(d["effective_date"]),
    }
    all_chunks += chunking.chunk_document(
        d["text"], d["doc_id"], strategy=CHUNK_STRATEGY,
        max_chars=S.chunk_max_chars, overlap_chars=S.chunk_overlap_chars,
        chunking_version=CHUNK_VERSION, metadata=base_meta,
    )

print(f"{len(all_chunks)} chunks (strategy={CHUNK_STRATEGY!r}, version={CHUNK_VERSION!r})")
print(f"avg chars/chunk: {sum(len(c.text) for c in all_chunks)//len(all_chunks)}")
print("\nsample chunk id :", all_chunks[0].id)
print("sample metadata :", {k: all_chunks[0].metadata[k] for k in ("doc_id","policy_category","chunk_index")})
print("sample text     :", all_chunks[0].text[:220], "…")

10 chunks (strategy='semantic', version='v1')
avg chars/chunk: 834

sample chunk id : HR-BEN-002-8248de8f90458304
sample metadata : {'doc_id': 'HR-BEN-002', 'policy_category': 'benefits', 'chunk_index': 0}
sample text     : Employee Benefits Policy
 Document HR-BEN-002    Category: benefits    Effective 2025-01-01    Applies to: all Kai Components store staff
1. Eligibility
Employees scheduled to work 30 or more hours per week are eligib …


## 3 · Embed with Titan v2 &nbsp;·&nbsp; `RUN_AWS`

One embedding per chunk, 1024-dim Titan v2 — the **same** model for ingest and query (a
mismatch silently wrecks recall). Offline we skip the Bedrock call and just report what would
be embedded.

In [4]:
if RUN_AWS:
    probe = embeddings.embed_text("How much paid parental leave do I get?", S)
    print("query embedding dim:", len(probe), "(expected", S.embedding_dim, ")")
    chunk_vecs = embeddings.embed_texts([c.text for c in all_chunks], S)
    print("embedded", len(chunk_vecs), "chunks")
else:
    chunk_vecs = None
    print(f"[offline] would embed {len(all_chunks)} chunks with {S.embedding_model_id} "
          f"→ {len(all_chunks)}×{S.embedding_dim} float32 vectors")

[offline] would embed 10 chunks with amazon.titan-embed-text-v2:0 → 10×1024 float32 vectors


## 4 · Create the S3 Vectors index + ingest &nbsp;·&nbsp; `RUN_AWS`

The `hr-policies` index schema is **IMMUTABLE** at creation: dimension, distance metric, and the
**non-filterable** key set (`vectors.HR_NON_FILTERABLE_KEYS` — the large `chunk_text` + a few
display fields) are fixed. Everything else stays filterable so retrieval can be scoped by
`policy_category` (see §5). Records are `{key, embedding, metadata}`; `put_records` batches ≤500
with backoff.

In [5]:
records = [
    {"key": c.id, "embedding": (chunk_vecs[i] if chunk_vecs else []),
     "metadata": {**c.metadata, "chunk_text": c.text}}
    for i, c in enumerate(all_chunks)
]
print("non-filterable keys (fixed at create):", vectors.HR_NON_FILTERABLE_KEYS)
filterable = [k for k in records[0]["metadata"] if k not in vectors.HR_NON_FILTERABLE_KEYS]
print("filterable metadata keys           :", filterable)

if RUN_AWS:
    vx = vectors.make_client(S.region)
    vectors.ensure_vector_bucket(vx, HR_BUCKET)
    vectors.ensure_index(vx, HR_BUCKET, HR_INDEX, dimension=S.embedding_dim,
                         distance_metric=S.distance_metric,
                         non_filterable_keys=vectors.HR_NON_FILTERABLE_KEYS)
    written = vectors.put_records(vx, HR_BUCKET, HR_INDEX, records, batch_size=S.ingest_batch_size)
    print(f"ingested {written} vectors into {HR_BUCKET}/{HR_INDEX}")
else:
    print(f"\n[offline] would create {HR_BUCKET}/{HR_INDEX} (dim={S.embedding_dim}, "
          f"metric={S.distance_metric}) and upsert {len(records)} records")

non-filterable keys (fixed at create): ['chunk_text', 'title', 'source_uri', 'effective_date']
filterable metadata keys           : ['policy_category', 'effective_epoch', 'doc_id', 'chunk_index', 'chunking_strategy', 'chunking_version']

[offline] would create dev-st21arbiter-poc-hr-vectors/hr-policies (dim=1024, metric=cosine) and upsert 10 records


## 5 · Retrieve — semantic search + a metadata-filtered search &nbsp;·&nbsp; `RUN_AWS`

Live retrieval embeds the question and runs nearest-neighbour search; a **metadata filter**
(`build_filter`) scopes results to one `policy_category` — the same mechanism used for
access-control / precision. **Offline** we stand in a tiny lexical retriever over the chunks so
the ranking logic is still exercised end-to-end without Bedrock.

In [6]:
import re
def _lexical_retrieve(question, chunks, k, category=None):
    """Offline stand-in: keyword-overlap score over chunk text (no embeddings)."""
    qtok = set(re.findall(r"[a-z0-9]+", question.lower()))
    scored = []
    for c in chunks:
        if category and c.metadata.get("policy_category") != category:
            continue
        ctok = set(re.findall(r"[a-z0-9]+", c.text.lower()))
        scored.append((len(qtok & ctok), c))
    scored.sort(key=lambda t: t[0], reverse=True)
    return [c for _, c in scored[:k]]

Q = "How much paid parental leave do employees get?"
print(f"Q: {Q}\n")
if RUN_AWS:
    for c in retrieval.retrieve(Q, HR_INDEX, S, top_k=TOP_K, use_rerank=USE_RERANK):
        print(f"  d={c.distance if c.distance is None else round(c.distance,4)}  {c.doc_id:14} {c.text[:80]}")
    print("\n-- filtered to policy_category='leave' --")
    flt = vectors.build_filter(equals={"policy_category": "leave"})
    for c in retrieval.retrieve("How much parental leave is paid?", HR_INDEX, S,
                                metadata_filter=flt, top_k=TOP_K, use_rerank=False):
        print(f"  {c.metadata.get('policy_category'):12} {c.doc_id:14} {c.text[:70]}")
else:
    for c in _lexical_retrieve(Q, all_chunks, TOP_K):
        print(f"  [lexical] {c.doc_id:14} {c.text[:80]}")
    print("\n-- filtered to policy_category='leave' (build_filter shown) --")
    print("  filter:", vectors.build_filter(equals={"policy_category": "leave"}))
    for c in _lexical_retrieve("How much parental leave is paid?", all_chunks, TOP_K, category="leave"):
        print(f"  [lexical] {c.doc_id:14} {c.text[:70]}")

Q: How much paid parental leave do employees get?

  [lexical] HR-LEAVE-001   Any
balance above 40 hours on December 31 is forfeited. Upon voluntary separatio
  [lexical] HR-LEAVE-001   Paid Time Off and Leave Policy
 Document HR-LEAVE-001    Category: leave    Ef
  [lexical] HR-PAY-005     Payroll and Scheduling Policy
 Document HR-PAY-005    Category: payroll    Eff
  [lexical] HR-BEN-002     Employee Benefits Policy
 Document HR-BEN-002    Category: benefits    Effecti

-- filtered to policy_category='leave' (build_filter shown) --
  filter: {'policy_category': {'$eq': 'leave'}}
  [lexical] HR-LEAVE-001   Any
balance above 40 hours on December 31 is forfeited. Upon voluntary
  [lexical] HR-LEAVE-001   Paid Time Off and Leave Policy
 Document HR-LEAVE-001    Category: le


## 6 · Rerank — before vs after &nbsp;·&nbsp; `RUN_AWS` + `USE_RERANK`

Reranking reorders the vector candidates with a cross-encoder for higher precision@k. It is
**off by default** (the deployed role has no `bedrock:Rerank` and it adds latency). Flip
`USE_RERANK = True` in cell 0 once you want to A/B it live.

In [7]:
Q2 = "What discount do employees get when buying products?"
if RUN_AWS and USE_RERANK:
    print("-- vector order (no rerank) --")
    for c in retrieval.retrieve(Q2, HR_INDEX, S, top_k=TOP_K, use_rerank=False):
        print(f"  d={round(c.distance,4) if c.distance is not None else None}  {c.doc_id:14} {c.text[:70]}")
    print("\n-- reranked order --")
    for c in retrieval.retrieve(Q2, HR_INDEX, S, top_k=TOP_K, use_rerank=True):
        print(f"  score={round(c.rerank_score,4) if c.rerank_score is not None else None}  {c.doc_id:14} {c.text[:70]}")
else:
    print(f"[skipped] set RUN_AWS=True and USE_RERANK=True to compare orderings for:\n  {Q2!r}")

[skipped] set RUN_AWS=True and USE_RERANK=True to compare orderings for:
  'What discount do employees get when buying products?'


## 7 · Grounded answer — **streaming + inline citations** &nbsp;·&nbsp; `RUN_AWS`

`retrieval.answer(..., stream=on_delta)` runs the full turn: guardrail(in) → retrieve →
**`generation.generate_stream`** (Bedrock `converse_stream`) → guardrail(out). Two features from
the shared `arbiter_rag` helper — the **same code the `hr_specialist` agent runs**:

- **(a) partial display** — each text delta is handed to `on_delta` as it arrives, so the answer
  prints live while the model is still generating; token usage comes from the stream's `metadata`.
- **(b) inline citations** — the model emits `[n]` markers against the numbered context passages,
  and `inject_citations` rewrites them next to the text as `[n](doc_id)` (+ a `citations` list).

In [8]:
from arbiter_rag import generation
Q3 = "How is sales commission calculated and when is it paid?"
print(f"Q: {Q3}\n")
if RUN_AWS:
    # (a) stream the partial answer as it forms
    def _show(piece): print(piece, end="", flush=True)
    ans = retrieval.answer(Q3, HR_INDEX, S, bucket=HR_BUCKET, system=None, stream=_show)
    print("\n\n-- final answer with inline citations (b) --")
    print(ans.answer)
    print("\ncitations:", [(c["marker"], c["doc_id"]) for c in ans.citations])
    print(f"tokens: in={ans.input_tokens} out={ans.output_tokens}  model={S.generation_model_id}")
else:
    # Offline: exercise the SAME citation-injection helper on a sample model output.
    ctx = _lexical_retrieve(Q3, all_chunks, TOP_K)
    sources = {i+1: {"doc_id": c.doc_id, "title": c.metadata.get("title","")} for i,c in enumerate(ctx)}
    print("[offline] retrieved context the LLM would answer from:")
    for i,c in enumerate(ctx,1): print(f"  [{i}] {c.doc_id} | {c.metadata.get('title')}")
    sample = "Commission is 4% of net sales above the monthly quota [1], paid on the 15th of the next month [1]."
    cited, cites = generation.inject_citations(sample, sources)
    print("\n[offline] inject_citations(sample_output) — feature (b):")
    print("  ", cited)
    print("  citations:", [(x["marker"], x["doc_id"]) for x in cites])

Q: How is sales commission calculated and when is it paid?

[offline] retrieved context the LLM would answer from:
  [1] HR-COMP-003 | Sales Compensation and Commission Policy
  [2] HR-CONDUCT-004 | Code of Conduct and Store Standards
  [3] HR-LEAVE-001 | Paid Time Off and Leave Policy
  [4] HR-LEAVE-001 | Paid Time Off and Leave Policy

[offline] inject_citations(sample_output) — feature (b):
   Commission is 4% of net sales above the monthly quota [1](HR-COMP-003), paid on the 15th of the next month [1](HR-COMP-003).
  citations: [(1, 'HR-COMP-003')]


## 8 · Evaluate retrieval against the golden set

`eval/golden/hr_qa.jsonl` holds 23 fictional-but-grounded Q&A (each tagged with the
`relevant_doc_ids` a good retrieval must surface). Offline we report golden **integrity** +
category coverage + a **lexical recall@k** proxy; live (`RUN_AWS`) we compute real
**recall@k / hit-rate** over the `hr-policies` index via `evaluation.aggregate_retrieval`.

In [9]:
cases = [json.loads(l) for l in (RAG_ROOT/"eval"/"golden"/"hr_qa.jsonl").read_text().splitlines() if l.strip()]
corpus_ids = {d["doc_id"] for d in docs}
missing = {rid for c in cases for rid in c["relevant_doc_ids"] if rid not in corpus_ids}
print(f"{len(cases)} golden cases | corpus covers relevant docs: {not missing}",
      ("MISSING: " + str(missing)) if missing else "")

if RUN_AWS:
    per = []
    for c in cases:
        ctxs = retrieval.retrieve(c["question"], HR_INDEX, S, top_k=TOP_K, use_rerank=USE_RERANK)
        per.append({"retrieved_ids": [x.doc_id for x in ctxs],
                    "relevant_ids": c["relevant_doc_ids"],
                    "top_distance": ctxs[0].distance if ctxs else None})
    report = evaluation.aggregate_retrieval(per, TOP_K)
    print("LIVE retrieval report:", report)
else:
    hits = 0
    for c in cases:
        got = {x.doc_id for x in _lexical_retrieve(c["question"], all_chunks, TOP_K)}
        if set(c["relevant_doc_ids"]) & got:
            hits += 1
    print(f"[offline] lexical hit-rate@{TOP_K}: {hits}/{len(cases)} = {hits/len(cases):.2f}")
    print("  (a floor; the live embedding retriever should meet or beat this)")

23 golden cases | corpus covers relevant docs: True 
[offline] lexical hit-rate@4: 23/23 = 1.00
  (a floor; the live embedding retriever should meet or beat this)


## 9 · Lifecycle — right-to-be-forgotten & cleanup &nbsp;·&nbsp; commented

`delete_document(doc_id)` removes every vector whose metadata `doc_id` matches (GDPR erasure);
`delete_keys` + `delete_index` tear the index down. Left commented so the lab is non-destructive.

In [10]:
# vx = vectors.make_client(S.region)
# n = vectors.delete_document(vx, HR_BUCKET, HR_INDEX, "HR-PERK-006")   # forget one policy
# vectors.delete_keys(vx, HR_BUCKET, HR_INDEX, [r["key"] for r in records])
# vx.delete_index(vectorBucketName=HR_BUCKET, indexName=HR_INDEX)
print("delete_document(doc_id) removes every vector for that policy; see rag_instructions.md for teardown.")

delete_document(doc_id) removes every vector for that policy; see rag_instructions.md for teardown.


## 10 · Where this goes → the `hr_specialist` agent

The streaming retrieve-then-answer path validated in §7 is exactly what
`agents/hr_specialist/agent.py` runs — it calls `retrieval.answer(..., stream=…)` (the same
`generate_stream` + `inject_citations` helper, no Strands tool-loop), so the agent streams and
returns inline-cited answers. To ship it: provision the index (`RUN_AWS=True` ingest here, or the
runbook's ingest step),
deploy with `scripts/deploy_agents.py --agents hr-specialist master-orchestrator`, then chat via
the **HR Specialist** card in the MCP page. Full steps: **`rag_instructions.md` (HR section)**.